# 12 — Content Based Book Recommendation System

## Objective

This notebook develops the content based recommendation system for the Leadership and Management Book Recommendation project.

The recommendation system will use the final NLP representation created in Notebook 09 and the validated topic clusters created in Notebook 11.

The initial recommendation approach will use cosine similarity to identify books with similar semantic content.

The recommendation workflow will:

1. Load the final clustered book catalogue.
2. Load the final enriched TF IDF representation.
3. Align TF IDF matrix rows with book IDs.
4. Calculate semantic similarity between books.
5. Return the most similar books for a selected title.
6. Incorporate topic cluster information to improve interpretation.
7. Test recommendations using several representative books.
8. Export the required artifacts for the Streamlit application.

This notebook focuses on recommendation logic. User interface development will be handled separately after the recommendation system has been validated.

In [16]:
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
PROJECT_DIR = Path(
    "/Users/jannoelvero/Documents/Ironhack/Week_11/"
    "Leadership_Management_Book_Recommendation_System"
)

PROCESSED_DIR = (
    PROJECT_DIR /
    "data" /
    "processed"
)

MODELS_DIR = (
    PROJECT_DIR /
    "models"
)

print("Project directory:", PROJECT_DIR)
print("Processed directory exists:", PROCESSED_DIR.exists())
print("Models directory exists:", MODELS_DIR.exists())

Project directory: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System
Processed directory exists: True
Models directory exists: True


In [18]:
CLUSTER_BOOKS_PATH = (
    PROCESSED_DIR /
    "books_with_final_topic_clusters.csv"
)

NLP_INDEX_PATH = (
    PROCESSED_DIR /
    "nlp_book_index.csv"
)

ENRICHED_MATRIX_PATH = (
    MODELS_DIR /
    "enriched_tfidf_matrix.npz"
)

ENRICHED_VECTORIZER_PATH = (
    MODELS_DIR /
    "enriched_tfidf_vectorizer.joblib"
)

In [19]:
print("Cluster books:", CLUSTER_BOOKS_PATH.exists())
print("NLP index:", NLP_INDEX_PATH.exists())
print("TF IDF matrix:", ENRICHED_MATRIX_PATH.exists())
print(
    "TF IDF vectorizer:",
    ENRICHED_VECTORIZER_PATH.exists()
)

Cluster books: True
NLP index: True
TF IDF matrix: True
TF IDF vectorizer: True


In [20]:
import re

from sklearn.feature_extraction.text import (
    ENGLISH_STOP_WORDS
)


FIELD_BOUNDARY = "zzfieldboundaryzz"

STOP_WORDS = set(
    ENGLISH_STOP_WORDS
)


def boundary_aware_analyzer(document):
    """
    Generate Unicode-aware unigrams and bigrams independently
    within each metadata field.

    Cross-field bigrams are prevented by processing each field
    separately.
    """

    features = []

    fields = document.split(
        FIELD_BOUNDARY
    )

    for field in fields:

        tokens = re.findall(
            r"(?u)\b[^\W_][\w'-]+\b",
            field.lower()
        )

        tokens = [
            token
            for token in tokens
            if token not in STOP_WORDS
        ]

        features.extend(tokens)

        features.extend(
            [
                f"{tokens[i]} {tokens[i + 1]}"
                for i in range(
                    len(tokens) - 1
                )
            ]
        )

    return features

In [21]:
books = pd.read_csv(
    CLUSTER_BOOKS_PATH
)

nlp_index = pd.read_csv(
    NLP_INDEX_PATH
)

enriched_tfidf = sparse.load_npz(
    ENRICHED_MATRIX_PATH
)

enriched_vectorizer = joblib.load(
    ENRICHED_VECTORIZER_PATH
)

In [22]:
print(
    "Clustered books:",
    books.shape
)

print(
    "NLP index:",
    nlp_index.shape
)

print(
    "Enriched TF IDF:",
    enriched_tfidf.shape
)

print(
    "Vectorizer loaded:",
    type(enriched_vectorizer)
)

Clustered books: (1884, 73)
NLP index: (2067, 6)
Enriched TF IDF: (2067, 5130)
Vectorizer loaded: <class 'sklearn.feature_extraction.text.TfidfVectorizer'>


In [23]:
print(
    "Books:",
    len(books)
)

print(
    "Unique book IDs:",
    books["book_id"].nunique()
)

print(
    "Topic clusters:",
    books["topic_cluster"].nunique()
)

print(
    "Cluster labels:",
    books["cluster_label"].nunique()
)

print(
    "Missing cluster labels:",
    books["cluster_label"].isna().sum()
)

Books: 1884
Unique book IDs: 1884
Topic clusters: 29
Cluster labels: 29
Missing cluster labels: 0


In [24]:
nlp_index.head()

,matrix_row,book_id,canonical_title,source_group,core_zero_vector,enriched_zero_vector
0,0,BOOK00001,Principle-Centered Leadership,Open Library only,False,False
1,1,BOOK00002,Leadership in Organizations,Open Library only,False,False
2,2,BOOK00003,Kepemimpinan =,Open Library only,True,False
3,3,BOOK00004,Spiritual leadership,Open Library only,False,False
4,4,BOOK00005,Leadership,Open Library only,False,False


In [10]:
print(
    "NLP index rows:",
    len(nlp_index)
)

print(
    "TF IDF matrix rows:",
    enriched_tfidf.shape[0]
)

print(
    "Unique NLP book IDs:",
    nlp_index["book_id"].nunique()
)

NLP index rows: 2067
TF IDF matrix rows: 2067
Unique NLP book IDs: 2067


In [11]:
print(
    "Matrix and index aligned:",
    len(nlp_index) == enriched_tfidf.shape[0]
)

Matrix and index aligned: True


In [25]:
# ============================================================
# COSINE SIMILARITY SETUP
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity


def get_book_similarities(book_index, tfidf_matrix):
    """
    Calculate cosine similarity between one selected book
    and all books in the TF-IDF matrix.
    """
    
    similarity_scores = cosine_similarity(
        tfidf_matrix[book_index],
        tfidf_matrix
    ).flatten()
    
    return similarity_scores


# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

test_book_index = 0

test_scores = get_book_similarities(
    test_book_index,
    enriched_tfidf
)

print("COSINE SIMILARITY VALIDATION")
print("=" * 80)

print(
    "Number of similarity scores:",
    len(test_scores)
)

print(
    "Self similarity:",
    test_scores[test_book_index]
)

print(
    "Minimum similarity:",
    test_scores.min()
)

print(
    "Maximum similarity:",
    test_scores.max()
)

assert len(test_scores) == 2067

assert np.isclose(
    test_scores[test_book_index],
    1.0
)

print(
    "\n✓ Cosine similarity calculation validated."
)

COSINE SIMILARITY VALIDATION
Number of similarity scores: 2067
Self similarity: 1.0
Minimum similarity: 0.0
Maximum similarity: 1.0

✓ Cosine similarity calculation validated.


In [26]:
# ============================================================
# TOP-N CONTENT-BASED RECOMMENDATION FUNCTION
# ============================================================

def recommend_by_book_id(
    book_id,
    n_recommendations=10
):
    """
    Return the Top-N most similar books using cosine similarity
    on the validated enriched TF-IDF representation.
    """

    # --------------------------------------------------------
    # Find query book in NLP index
    # --------------------------------------------------------
    
    matches = nlp_index.index[
        nlp_index["book_id"] == book_id
    ].tolist()

    if not matches:
        raise ValueError(
            f"Book ID not found: {book_id}"
        )

    query_index = matches[0]

    # --------------------------------------------------------
    # Calculate similarity
    # --------------------------------------------------------
    
    similarity_scores = get_book_similarities(
        query_index,
        enriched_tfidf
    )

    # Sort from highest to lowest similarity
    ranked_indices = np.argsort(
        similarity_scores
    )[::-1]

    # Remove query book itself
    ranked_indices = [
        idx
        for idx in ranked_indices
        if idx != query_index
    ]

    # Keep Top-N
    top_indices = ranked_indices[
        :n_recommendations
    ]

    # --------------------------------------------------------
    # Build recommendation table
    # --------------------------------------------------------
    
    recommendations = (
        nlp_index
        .iloc[top_indices]
        .copy()
    )

    recommendations[
        "similarity_score"
    ] = similarity_scores[
        top_indices
    ]

    recommendations[
        "recommendation_rank"
    ] = range(
        1,
        len(recommendations) + 1
    )

    return recommendations


print("✓ Top-N recommendation function created.")

✓ Top-N recommendation function created.


In [27]:
# ============================================================
# TEST FIRST RECOMMENDATION
# ============================================================

test_book_id = nlp_index.iloc[0]["book_id"]

print("Query book ID:", test_book_id)

test_recommendations = recommend_by_book_id(
    test_book_id,
    n_recommendations=10
)

display(
    test_recommendations[
        [
            "recommendation_rank",
            "book_id",
            "similarity_score"
        ]
    ]
)

Query book ID: BOOK00001


,recommendation_rank,book_id,similarity_score
332,1,BOOK00333,0.220669
1915,2,BOOK01916,0.208324
27,3,BOOK00028,0.198383
904,4,BOOK00905,0.177451
1901,5,BOOK01902,0.160279
57,6,BOOK00058,0.159162
207,7,BOOK00208,0.156437
1965,8,BOOK01966,0.144820
863,9,BOOK00864,0.139195
808,10,BOOK00809,0.133814


In [28]:
# ============================================================
# INSPECT AVAILABLE BOOK METADATA
# ============================================================

print("BOOK DATASET")
print("=" * 80)

print("Shape:", books.shape)

print("\nAvailable columns:")
for column in books.columns:
    print("-", column)

print("\nFirst 3 records:")
display(
    books.head(3)
)

BOOK DATASET
Shape: (1884, 73)

Available columns:
- book_id
- canonical_title
- authors
- description
- subjects
- first_publish_year
- average_rating
- ratings_count
- edition_count
- want_to_read_count
- currently_reading_count
- already_read_count
- cover_url
- openlibrary_key
- source_openlibrary
- source_leadershipnow
- publication_year_observed
- title_normalized
- has_authors
- has_description
- has_subjects
- has_cover
- has_rating
- has_engagement
- has_extended_text
- has_first_publish_year
- has_observed_publication_year
- book_age_from_first_publish
- years_since_observed_publication
- log1p_ratings_count
- log1p_want_to_read_count
- log1p_currently_reading_count
- log1p_already_read_count
- log1p_edition_count
- total_reader_engagement
- log1p_total_reader_engagement
- format
- page_count
- has_edition_metadata
- has_page_count
- is_hardcover
- is_paperback
- log1p_page_count
- format_edition
- page_count_edition
- has_edition_metadata_edition
- has_page_count_edition
- i

,book_id,canonical_title,authors,description,subjects,first_publish_year,average_rating,ratings_count,edition_count,want_to_read_count,...,source_group,core_content_text,enriched_content_text,core_word_count,enriched_word_count,additional_semantic_words,topic_content_text,topic_cluster,topic_text_clean,cluster_label
0,BOOK00001,Principle-Centered Leadership,['Stephen R. Covey'],How do we as individuals and organizations sur...,"['Leadership', 'Psychological aspects of Succe...",1989.0,4.5,2.0,21.0,145.0,...,Open Library only,Principle-Centered Leadership Stephen R. Covey,Principle-Centered Leadership Stephen R. Covey...,5,199,194,Principle-Centered Leadershipzzfieldboundaryzz...,7,Principle-Centered Leadership Leadership Psych...,Broad Leadership and Success
1,BOOK00002,Leadership in Organizations,['Gary A. Yukl'],NaN,"['Organisation', 'Prise de décision', 'Entsche...",1981.0,5.0,1.0,26.0,110.0,...,Open Library only,Leadership in Organizations Gary A. Yukl,Leadership in Organizations Gary A. Yukl Organ...,6,51,45,Leadership in OrganizationszzfieldboundaryzzOr...,4,Leadership in Organizations Organisation Prise...,Decision Making
2,BOOK00003,Kepemimpinan =,['Karjadi M.'],NaN,['Leadership'],1977.0,1.0,1.0,1.0,41.0,...,Open Library only,Kepemimpinan = Karjadi M.,Kepemimpinan = Karjadi M. Leadership,4,5,1,Kepemimpinan =zzfieldboundaryzzLeadershipzzfie...,25,Kepemimpinan = Leadership,Transformational Leadership


In [30]:
# ============================================================
# CHECK COLUMNS AFTER METADATA MERGE
# ============================================================

print("Columns in test_recommendations:")
print(test_recommendations.columns.tolist())

print("\nColumns after metadata merge:")
print(test_recommendations_enriched.columns.tolist())

Columns in test_recommendations:
['matrix_row', 'book_id', 'canonical_title', 'source_group', 'core_zero_vector', 'enriched_zero_vector', 'similarity_score', 'recommendation_rank']

Columns after metadata merge:
['matrix_row', 'book_id', 'canonical_title_x', 'source_group_x', 'core_zero_vector', 'enriched_zero_vector', 'similarity_score', 'recommendation_rank', 'canonical_title_y', 'authors', 'cluster_label', 'topic_cluster', 'source_group_y', 'first_publish_year', 'average_rating', 'ratings_count']


In [31]:
# ============================================================
# ENRICH RECOMMENDATIONS WITH BOOK METADATA
# ============================================================

def enrich_recommendations(recommendations):
    """
    Add additional book metadata and final topic-cluster
    information to recommendation results.

    canonical_title and source_group are already available
    from nlp_index, so they are not merged again.

    A left join preserves all recommendation candidates,
    including books that were not included in topic clustering.
    """

    metadata_columns = [
        "book_id",
        "authors",
        "cluster_label",
        "topic_cluster",
        "first_publish_year",
        "average_rating",
        "ratings_count"
    ]

    enriched = recommendations.merge(
        books[metadata_columns],
        on="book_id",
        how="left",
        validate="one_to_one"
    )

    return enriched


# ------------------------------------------------------------
# Enrich first recommendation test
# ------------------------------------------------------------

test_recommendations_enriched = enrich_recommendations(
    test_recommendations
)


# ------------------------------------------------------------
# Validate
# ------------------------------------------------------------

print("ENRICHED RECOMMENDATION VALIDATION")
print("=" * 80)

print(
    "Recommendations:",
    len(test_recommendations_enriched)
)

print(
    "Duplicate book IDs:",
    test_recommendations_enriched["book_id"].duplicated().sum()
)

print(
    "Missing titles:",
    test_recommendations_enriched["canonical_title"].isna().sum()
)

print(
    "Missing cluster labels:",
    test_recommendations_enriched["cluster_label"].isna().sum()
)


# ------------------------------------------------------------
# Display readable recommendations
# ------------------------------------------------------------

display(
    test_recommendations_enriched[
        [
            "recommendation_rank",
            "canonical_title",
            "authors",
            "cluster_label",
            "similarity_score"
        ]
    ]
)

ENRICHED RECOMMENDATION VALIDATION
Recommendations: 10
Duplicate book IDs: 0
Missing titles: 0
Missing cluster labels: 0


,recommendation_rank,canonical_title,authors,cluster_label,similarity_score
0,1,The 7 Habits of Highly Effective People,['Stephen R. Covey'],Broad Leadership and Success,0.220669
1,2,Live Life in Crescendo,['Stephen R. Covey and Cynthia Covey'],CEO and Executive Transformation,0.208324
2,3,The Tao of leadership,['John Heider'],Transformational Leadership,0.198383
3,4,Primal Leadership,"['Daniel Goleman', 'Richard E. Boyatzis', 'Ann...",Emotional Intelligence,0.177451
4,5,Trust and Inspire,['Stephen M.R. Covey'],Organizational Culture and Trust,0.160279
5,6,Leadership development,['Rosemary Ryan'],Leadership Development,0.159162
6,7,7 principles of transformational leadership,['Hugh Blane'],Broad Leadership and Success,0.156437
7,8,Life on the X,['Stephen Drum'],CEO and Executive Transformation,0.144820
8,9,Emotional intelligence,"['Walton, David (Psychologist)']",Emotional Intelligence,0.139195
9,10,Nonviolent Communication,"['Marshall B. Rosenberg', 'Deepak Chopra', 'Ma...",Communication,0.133814


In [32]:
# ============================================================
# SEARCH BOOKS BY TITLE
# ============================================================

def search_books_by_title(
    title_query,
    max_results=10
):
    """
    Search available books using a case-insensitive
    partial title match.
    """

    query = str(title_query).strip()

    if not query:
        return pd.DataFrame()

    matches = nlp_index[
        nlp_index["canonical_title"]
        .fillna("")
        .str.contains(
            query,
            case=False,
            regex=False
        )
    ].copy()

    return (
        matches[
            [
                "book_id",
                "canonical_title",
                "source_group"
            ]
        ]
        .head(max_results)
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# Test title search
# ------------------------------------------------------------

title_search_test = search_books_by_title(
    "Principle-Centered Leadership"
)

print("TITLE SEARCH TEST")
print("=" * 80)
print(
    "Matches found:",
    len(title_search_test)
)

display(title_search_test)

TITLE SEARCH TEST
Matches found: 1


,book_id,canonical_title,source_group
0,BOOK00001,Principle-Centered Leadership,Open Library only


In [34]:
# ============================================================
# USER-FACING BOOK RECOMMENDATION FUNCTION
# ============================================================

def recommend_books(
    title,
    n_recommendations=10
):
    """
    Recommend books from a user-provided book title.

    Workflow:
    1. Search for the title.
    2. Identify the corresponding book_id.
    3. Calculate TF-IDF cosine similarity.
    4. Retrieve Top-N recommendations.
    5. Add readable metadata and topic-cluster information.
    """

    # --------------------------------------------------------
    # Search for title
    # --------------------------------------------------------

    matches = search_books_by_title(
        title,
        max_results=10
    )

    if matches.empty:
        print(
            f'No book found matching "{title}".'
        )
        return pd.DataFrame()

    # --------------------------------------------------------
    # Use first matching book
    # --------------------------------------------------------

    selected_book = matches.iloc[0]

    selected_book_id = selected_book[
        "book_id"
    ]

    selected_title = selected_book[
        "canonical_title"
    ]

    # --------------------------------------------------------
    # Generate recommendations
    # --------------------------------------------------------

    recommendations = recommend_by_book_id(
        selected_book_id,
        n_recommendations=n_recommendations
    )

    recommendations = enrich_recommendations(
        recommendations
    )

    # --------------------------------------------------------
    # Display selected book
    # --------------------------------------------------------

    print("BOOK RECOMMENDATION SYSTEM")
    print("=" * 80)

    print(
        "Selected book:",
        selected_title
    )

    print(
        "Book ID:",
        selected_book_id
    )

    print(
        "Recommendations:",
        len(recommendations)
    )

    # --------------------------------------------------------
    # Return useful output columns
    # --------------------------------------------------------

    output_columns = [
        "recommendation_rank",
        "book_id",
        "canonical_title",
        "authors",
        "cluster_label",
        "similarity_score"
    ]

    return (
        recommendations[
            output_columns
        ]
        .reset_index(drop=True)
    )


print(
    "✓ User-facing recommendation function created."
)

✓ User-facing recommendation function created.


In [35]:
recommend_books(
    "Principle-Centered Leadership",
    n_recommendations=10
)

BOOK RECOMMENDATION SYSTEM
Selected book: Principle-Centered Leadership
Book ID: BOOK00001
Recommendations: 10


,recommendation_rank,book_id,canonical_title,authors,cluster_label,similarity_score
0,1,BOOK00333,The 7 Habits of Highly Effective People,['Stephen R. Covey'],Broad Leadership and Success,0.220669
1,2,BOOK01916,Live Life in Crescendo,['Stephen R. Covey and Cynthia Covey'],CEO and Executive Transformation,0.208324
2,3,BOOK00028,The Tao of leadership,['John Heider'],Transformational Leadership,0.198383
3,4,BOOK00905,Primal Leadership,"['Daniel Goleman', 'Richard E. Boyatzis', 'Ann...",Emotional Intelligence,0.177451
4,5,BOOK01902,Trust and Inspire,['Stephen M.R. Covey'],Organizational Culture and Trust,0.160279
5,6,BOOK00058,Leadership development,['Rosemary Ryan'],Leadership Development,0.159162
6,7,BOOK00208,7 principles of transformational leadership,['Hugh Blane'],Broad Leadership and Success,0.156437
7,8,BOOK01966,Life on the X,['Stephen Drum'],CEO and Executive Transformation,0.144820
8,9,BOOK00864,Emotional intelligence,"['Walton, David (Psychologist)']",Emotional Intelligence,0.139195
9,10,BOOK00809,Nonviolent Communication,"['Marshall B. Rosenberg', 'Deepak Chopra', 'Ma...",Communication,0.133814


In [36]:
# ============================================================
# FIND TEST BOOKS ACROSS DIFFERENT MANAGEMENT THEMES
# ============================================================

test_queries = [
    "emotional intelligence",
    "project management",
    "servant leadership",
    "change management",
    "human resource",
    "strategy"
]

for query in test_queries:
    
    matches = search_books_by_title(
        query,
        max_results=5
    )
    
    print("\n" + "=" * 80)
    print("SEARCH:", query)
    print("=" * 80)
    
    if matches.empty:
        print("No matches found.")
    else:
        display(matches)


SEARCH: emotional intelligence


,book_id,canonical_title,source_group
0,BOOK00856,Emotional Intelligence,Open Library only
1,BOOK00857,Emotional Intelligence 2.0,Open Library only
2,BOOK00858,Emotional Intelligence,Open Library only
3,BOOK00859,Working with Emotional Intelligence,Open Library only
4,BOOK00860,Emotional Intelligence,Open Library only



SEARCH: project management


,book_id,canonical_title,source_group
0,BOOK00299,Project management,Open Library only
1,BOOK00309,A Guide to the Project Management Body of Know...,Open Library only
2,BOOK00330,Project management,Open Library only
3,BOOK00519,Project management for dummies,Open Library only
4,BOOK00520,Project Management,Open Library only



SEARCH: servant leadership


...

,book_id,canonical_title,source_group
0,BOOK00424,Change management,Open Library only
1,BOOK00426,Change management,Open Library only
2,BOOK00427,Change Management,Open Library only
3,BOOK00429,Change management in information services,Open Library only
4,BOOK00430,Making sense of change management,Open Library only



SEARCH: human resource


,book_id,canonical_title,source_group
0,BOOK00284,Human Resource management,Open Library only
1,BOOK00294,Human resource management,Open Library only
2,BOOK00298,Human Resource Management,Open Library only
3,BOOK00334,Armstrong's Handbook of Human Resource Managem...,Open Library only
4,BOOK00336,Human resource management at work,Open Library only



SEARCH: strategy


,book_id,canonical_title,source_group
0,BOOK00035,Leadership Strategy and Tactics,Open Library only
1,BOOK00410,Strategy safari,Open Library only
2,BOOK00708,Managerial Economics & Business Strategy,Open Library only
3,BOOK00712,Business strategy,Open Library only
4,BOOK00713,International business strategy,Open Library only


In [37]:
# ============================================================
# MULTI-THEME RECOMMENDATION TEST
# ============================================================

evaluation_books = {
    "Emotional Intelligence": "BOOK00856",
    "Project Management": "BOOK00299",
    "Servant Leadership": "BOOK00070",
    "Change Management": "BOOK00424",
    "Human Resource Management": "BOOK00284",
    "Strategy": "BOOK00410"
}

evaluation_results = {}

for theme, book_id in evaluation_books.items():

    # --------------------------------------------------------
    # Query book information
    # --------------------------------------------------------

    query_row = nlp_index[
        nlp_index["book_id"] == book_id
    ]

    assert len(query_row) == 1

    query_title = query_row.iloc[0][
        "canonical_title"
    ]

    # --------------------------------------------------------
    # Generate recommendations
    # --------------------------------------------------------

    recommendations = recommend_by_book_id(
        book_id,
        n_recommendations=5
    )

    recommendations = enrich_recommendations(
        recommendations
    )

    evaluation_results[theme] = recommendations

    # --------------------------------------------------------
    # Display
    # --------------------------------------------------------

    print("\n" + "=" * 90)
    print("THEME:", theme)
    print("QUERY BOOK:", query_title)
    print("BOOK ID:", book_id)
    print("=" * 90)

    display(
        recommendations[
            [
                "recommendation_rank",
                "canonical_title",
                "authors",
                "cluster_label",
                "similarity_score"
            ]
        ].reset_index(drop=True)
    )


THEME: Emotional Intelligence
QUERY BOOK: Emotional Intelligence
BOOK ID: BOOK00856


,recommendation_rank,canonical_title,authors,cluster_label,similarity_score
0,1,Working with Emotional Intelligence,['Daniel Goleman'],Emotional Intelligence,0.388770
1,2,Raising your emotional intelligence,['Jeanne Segal'],Emotional Intelligence,0.341676
2,3,La iInteligencia emocional,['Daniel Goleman'],Emotional Intelligence,0.313741
3,4,Emotional intelligence,"['Ralf Schulze', 'Richard D. Roberts']",Emotional Intelligence,0.250816
4,5,Emotional Intelligence,['Margaret Chapman'],Emotional Intelligence,0.250499



THEME: Project Management
QUERY BOOK: Project management
BOOK ID: BOOK00299


,recommendation_rank,canonical_title,authors,cluster_label,similarity_score
0,1,Project Management,['Harvey Maylor'],Project Management,0.364959
1,2,Successful project management,"['Jack Gido', 'James P. Clements']",Project Management,0.339655
2,3,Project management,['Harold Kerzner'],Project Management,0.277144
3,4,Project management,"['Jack R. Meredith', 'Samuel J. Mantel']",Project Management,0.266144
4,5,Project management,"['David L. Cleland', 'Lewis R. Ireland']",Project Management,0.263706



THEME: Servant Leadership
QUERY BOOK: Servant Leadership
BOOK ID: BOOK00070


...

,recommendation_rank,canonical_title,authors,cluster_label,similarity_score
0,1,The complete idiot's guide to change management,['Jeffrey P. Davidson'],Change Management,0.697930
1,2,Strategic Change Management Strategic Change M...,['Djamel Eddine Laouisset'],Change Management,0.525333
2,3,Change Management,"['Klaus Doppler', 'Christoph Lauterburg']",Change Management,0.435199
3,4,Enterprise Change Management,"['David Miller', 'Audra Proctor']",Change Management,0.403352
4,5,Leadership and change management,['Annabel C. Beerel'],Change Management,0.402510



THEME: Human Resource Management
QUERY BOOK: Human Resource management
BOOK ID: BOOK00284


,recommendation_rank,canonical_title,authors,cluster_label,similarity_score
0,1,Human Resource Management,['Gary Dessler'],Human Resource Management,1.000000
1,2,Human resource management,['Gary Dessler'],Human Resource Management,1.000000
2,3,Fundamentals of Human Resource Management,['Gary Dessler'],Human Resource Management,0.853093
3,4,Human resource management,['Gary Dessler'],Human Resource Management,0.851611
4,5,"Human Resource Management, 16th edition",['Gary Dessler'],Human Resource Management,0.713645



THEME: Strategy
QUERY BOOK: Strategy safari
BOOK ID: BOOK00410


,recommendation_rank,canonical_title,authors,cluster_label,similarity_score
0,1,Understanding Organizations...Finally!,['Henry Mintzberg'],Broad Leadership and Success,0.317119
1,2,Understanding Strategic Management,['Anthony Henry'],Strategic Management and Planning,0.262817
2,3,Strategic management,['Fred R. David'],Strategic Management and Planning,0.243522
3,4,Business strategy,['John Grieve Smith'],Strategic Management and Planning,0.228241
4,5,Strategic management,['Richard L. Lynch'],Strategic Management and Planning,0.220004


In [39]:
# ============================================================
# INVESTIGATE HR RECOMMENDATIONS FOR DUPLICATE / EDITION EFFECTS
# ============================================================

# Query book
hr_query_id = evaluation_books[
    "Human Resource Management"
]

# Recommended books from our completed evaluation
hr_recommendations = evaluation_results[
    "Human Resource Management"
]

# Query + recommendation IDs
hr_book_ids = [
    hr_query_id
] + hr_recommendations[
    "book_id"
].tolist()


# ------------------------------------------------------------
# Inspect available metadata
# ------------------------------------------------------------

hr_duplicate_check = books[
    books["book_id"].isin(hr_book_ids)
][
    [
        "book_id",
        "canonical_title",
        "authors",
        "first_publish_year",
        "subjects",
        "description",
        "source_group",
        "core_content_text",
        "enriched_content_text"
    ]
].copy()


print("HR DUPLICATE / EDITION INVESTIGATION")
print("=" * 80)

print(
    "Books inspected:",
    len(hr_duplicate_check)
)

display(
    hr_duplicate_check
    .sort_values(
        ["canonical_title", "book_id"]
    )
    .reset_index(drop=True)
)

HR DUPLICATE / EDITION INVESTIGATION
Books inspected: 6


,book_id,canonical_title,authors,first_publish_year,subjects,description,source_group,core_content_text,enriched_content_text
0,BOOK00633,Fundamentals of Human Resource Management,['Gary Dessler'],2008.0,['Personnel management'],NaN,Open Library only,Fundamentals of Human Resource Management Gary...,Fundamentals of Human Resource Management Gary...
1,BOOK00618,Human Resource Management,['Gary Dessler'],1999.0,['Personnel management'],NaN,Open Library only,Human Resource Management Gary Dessler,Human Resource Management Gary Dessler Personn...
2,BOOK00645,"Human Resource Management, 16th edition",['Gary Dessler'],2015.0,[],NaN,Open Library only,"Human Resource Management, 16th edition Gary D...","Human Resource Management, 16th edition Gary D..."
3,BOOK00284,Human Resource management,['Gary Dessler'],1994.0,['Personnel management'],NaN,Open Library only,Human Resource management Gary Dessler,Human Resource management Gary Dessler Personn...
4,BOOK00626,Human resource management,['Gary Dessler'],2005.0,"['Personeelsmanagement', 'Personnel management']",NaN,Open Library only,Human resource management Gary Dessler,Human resource management Gary Dessler Persone...
5,BOOK00652,Human resource management,['Gary Dessler'],2006.0,['Personnel management'],NaN,Open Library only,Human resource management Gary Dessler,Human resource management Gary Dessler Personn...


In [40]:
# ============================================================
# NORMALIZE TITLE + AUTHOR FOR RECOMMENDATION DEDUPLICATION
# ============================================================

def normalize_recommendation_text(value):
    """
    Normalize text for duplicate/edition comparison only.
    Does not modify the original book metadata.
    """
    
    if pd.isna(value):
        return ""
    
    value = str(value).lower().strip()
    
    # Keep letters and numbers; remove punctuation differences
    value = re.sub(r"[^\w\s]", " ", value)
    
    # Collapse repeated whitespace
    value = re.sub(r"\s+", " ", value).strip()
    
    return value


# ------------------------------------------------------------
# Create duplicate-comparison keys
# ------------------------------------------------------------

books["recommendation_title_key"] = (
    books["canonical_title"]
    .apply(normalize_recommendation_text)
)

books["recommendation_author_key"] = (
    books["authors"]
    .apply(normalize_recommendation_text)
)


print("✓ Recommendation duplicate-comparison keys created.")

display(
    books[
        books["book_id"].isin(hr_book_ids)
    ][
        [
            "book_id",
            "canonical_title",
            "authors",
            "recommendation_title_key",
            "recommendation_author_key"
        ]
    ]
    .sort_values("recommendation_title_key")
    .reset_index(drop=True)
)

✓ Recommendation duplicate-comparison keys created.


,book_id,canonical_title,authors,recommendation_title_key,recommendation_author_key
0,BOOK00633,Fundamentals of Human Resource Management,['Gary Dessler'],fundamentals of human resource management,gary dessler
1,BOOK00284,Human Resource management,['Gary Dessler'],human resource management,gary dessler
2,BOOK00618,Human Resource Management,['Gary Dessler'],human resource management,gary dessler
3,BOOK00626,Human resource management,['Gary Dessler'],human resource management,gary dessler
4,BOOK00652,Human resource management,['Gary Dessler'],human resource management,gary dessler
5,BOOK00645,"Human Resource Management, 16th edition",['Gary Dessler'],human resource management 16th edition,gary dessler


In [41]:
# ============================================================
# TOP-N RECOMMENDATION FUNCTION
# WITH SAME-TITLE + SAME-AUTHOR SUPPRESSION
# ============================================================

def recommend_by_book_id(
    book_id,
    n_recommendations=10
):
    """
    Generate content-based book recommendations using
    enriched TF-IDF cosine similarity.

    Rules:
    - Exclude the query book itself.
    - Exclude zero-similarity candidates.
    - Suppress repeated title + author combinations.
    - Preserve distinct books by the same author.
    """

    # --------------------------------------------------------
    # Locate query book
    # --------------------------------------------------------

    matches = nlp_index.index[
        nlp_index["book_id"] == book_id
    ].tolist()

    if not matches:
        raise ValueError(
            f"Book ID not found: {book_id}"
        )

    query_index = matches[0]


    # --------------------------------------------------------
    # Handle zero-vector query
    # --------------------------------------------------------

    if enriched_tfidf[query_index].nnz == 0:
        return pd.DataFrame(
            columns=[
                *nlp_index.columns,
                "similarity_score",
                "recommendation_rank"
            ]
        )


    # --------------------------------------------------------
    # Calculate similarities
    # --------------------------------------------------------

    similarity_scores = get_book_similarities(
        query_index,
        enriched_tfidf
    )

    ranked_indices = np.argsort(
        similarity_scores
    )[::-1]


    # --------------------------------------------------------
    # Query identity for duplicate suppression
    # --------------------------------------------------------

    query_book = books[
        books["book_id"] == book_id
    ]

    if not query_book.empty:

        query_title_key = query_book.iloc[0][
            "recommendation_title_key"
        ]

        query_author_key = query_book.iloc[0][
            "recommendation_author_key"
        ]

        query_duplicate_key = (
            query_title_key,
            query_author_key
        )

    else:
        query_duplicate_key = None


    # --------------------------------------------------------
    # Select unique positive-similarity recommendations
    # --------------------------------------------------------

    selected_indices = []
    seen_keys = set()

    for idx in ranked_indices:

        # Exclude query itself
        if idx == query_index:
            continue

        # Do not recommend books with no content similarity
        if similarity_scores[idx] <= 0:
            continue

        candidate_id = nlp_index.iloc[idx][
            "book_id"
        ]

        candidate_book = books[
            books["book_id"] == candidate_id
        ]

        # --------------------------------------------
        # Duplicate suppression when metadata exists
        # --------------------------------------------

        if not candidate_book.empty:

            candidate_title_key = candidate_book.iloc[0][
                "recommendation_title_key"
            ]

            candidate_author_key = candidate_book.iloc[0][
                "recommendation_author_key"
            ]

            candidate_key = (
                candidate_title_key,
                candidate_author_key
            )

            # Same logical title + author as query
            if (
                query_duplicate_key is not None
                and candidate_key == query_duplicate_key
            ):
                continue

            # Same logical title + author already selected
            if candidate_key in seen_keys:
                continue

            seen_keys.add(candidate_key)

        selected_indices.append(idx)

        if len(selected_indices) >= n_recommendations:
            break


    # --------------------------------------------------------
    # Build recommendation dataframe
    # --------------------------------------------------------

    recommendations = (
        nlp_index
        .iloc[selected_indices]
        .copy()
    )

    recommendations[
        "similarity_score"
    ] = similarity_scores[
        selected_indices
    ]

    recommendations[
        "recommendation_rank"
    ] = range(
        1,
        len(recommendations) + 1
    )

    return recommendations


print(
    "✓ Recommendation function upgraded with duplicate suppression."
)

✓ Recommendation function upgraded with duplicate suppression.


In [42]:
recommend_books(
    "Human Resource management",
    n_recommendations=10
)

BOOK RECOMMENDATION SYSTEM
Selected book: Human Resource management
Book ID: BOOK00284
Recommendations: 10


,recommendation_rank,book_id,canonical_title,authors,cluster_label,similarity_score
0,1,BOOK00633,Fundamentals of Human Resource Management,['Gary Dessler'],Human Resource Management,0.853093
1,2,BOOK00645,"Human Resource Management, 16th edition",['Gary Dessler'],Human Resource Management,0.713645
2,3,BOOK00658,Human resource management,['Laura Portolese Dias'],Human Resource Management,0.547271
3,4,BOOK00647,Human Resource Management,"['Ronan Carbery', 'Christine Cross']",Human Resource Management,0.545621
4,5,BOOK00636,Human Resource Management,['K. Aswathappa'],Human Resource Management,0.522556
5,6,BOOK00649,Human resource management,['Robert L. Mathis'],Human Resource Management,0.518366
6,7,BOOK00650,Human Resource Management,['Raymond A. Noe'],Human Resource Management,0.503095
7,8,BOOK00654,Human Resource Management,['Aswathappa'],Human Resource Management,0.486634
8,9,BOOK00634,Human resource management,['Cynthia D. Fisher'],Human Resource Management,0.475458
9,10,BOOK00643,Strategic Human Resource Management,"['Gary Rees', 'Paul E Smith']",Human Resource Management,0.474664


In [43]:
# ============================================================
# RECOMMENDATION SYSTEM INTEGRITY TESTS
# ============================================================

test_book_ids = [
    "BOOK00001",  # Principle-Centered Leadership
    "BOOK00856",  # Emotional Intelligence
    "BOOK00299",  # Project Management
    "BOOK00070",  # Servant Leadership
    "BOOK00424",  # Change Management
    "BOOK00284",  # Human Resource Management
    "BOOK00410"   # Strategy Safari
]

integrity_results = []

for book_id in test_book_ids:

    recommendations = recommend_by_book_id(
        book_id,
        n_recommendations=10
    )

    scores = recommendations[
        "similarity_score"
    ].to_numpy()

    # --------------------------------------------
    # Technical validation checks
    # --------------------------------------------

    self_excluded = (
        book_id not in
        recommendations["book_id"].values
    )

    unique_book_ids = (
        recommendations["book_id"].is_unique
    )

    positive_similarity = (
        (scores > 0).all()
        if len(scores) > 0
        else True
    )

    descending_order = (
        np.all(scores[:-1] >= scores[1:])
        if len(scores) > 1
        else True
    )

    requested_n_or_less = (
        len(recommendations) <= 10
    )

    integrity_results.append(
        {
            "book_id": book_id,
            "recommendations_returned":
                len(recommendations),
            "self_excluded":
                self_excluded,
            "unique_book_ids":
                unique_book_ids,
            "positive_similarity":
                positive_similarity,
            "descending_similarity":
                descending_order,
            "requested_n_or_less":
                requested_n_or_less
        }
    )


integrity_results_df = pd.DataFrame(
    integrity_results
)

print("RECOMMENDATION SYSTEM INTEGRITY TEST")
print("=" * 80)

display(integrity_results_df)


# ------------------------------------------------------------
# Overall validation
# ------------------------------------------------------------

validation_columns = [
    "self_excluded",
    "unique_book_ids",
    "positive_similarity",
    "descending_similarity",
    "requested_n_or_less"
]

all_tests_passed = (
    integrity_results_df[
        validation_columns
    ]
    .all()
    .all()
)

print(
    "\nAll integrity tests passed:",
    all_tests_passed
)

RECOMMENDATION SYSTEM INTEGRITY TEST


,book_id,recommendations_returned,self_excluded,unique_book_ids,positive_similarity,descending_similarity,requested_n_or_less
0,BOOK00001,10,True,True,True,True,True
1,BOOK00856,10,True,True,True,True,True
2,BOOK00299,10,True,True,True,True,True
3,BOOK00070,10,True,True,True,True,True
4,BOOK00424,10,True,True,True,True,True
5,BOOK00284,10,True,True,True,True,True
6,BOOK00410,10,True,True,True,True,True



All integrity tests passed: True


In [44]:
# ============================================================
# RECOMMENDATION CATALOG COVERAGE @ 10
# ============================================================

TOP_K = 10

eligible_query_indices = np.where(
    enriched_tfidf.getnnz(axis=1) > 0
)[0]

recommended_book_ids = set()
total_recommendations = 0

for query_index in eligible_query_indices:

    query_book_id = nlp_index.iloc[
        query_index
    ]["book_id"]

    recommendations = recommend_by_book_id(
        query_book_id,
        n_recommendations=TOP_K
    )

    recommended_book_ids.update(
        recommendations["book_id"].tolist()
    )

    total_recommendations += len(
        recommendations
    )


# ------------------------------------------------------------
# Calculate coverage
# ------------------------------------------------------------

catalog_size = len(nlp_index)

unique_recommended_books = len(
    recommended_book_ids
)

catalog_coverage = (
    unique_recommended_books /
    catalog_size
)


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("RECOMMENDATION CATALOG COVERAGE @ 10")
print("=" * 80)

print(
    "Total catalog books:",
    catalog_size
)

print(
    "Eligible query books:",
    len(eligible_query_indices)
)

print(
    "Total recommendation slots generated:",
    total_recommendations
)

print(
    "Unique books recommended:",
    unique_recommended_books
)

print(
    f"Catalog coverage: {catalog_coverage:.2%}"
)

RECOMMENDATION CATALOG COVERAGE @ 10
Total catalog books: 2067
Eligible query books: 2040
Total recommendation slots generated: 19396
Unique books recommended: 2016
Catalog coverage: 97.53%


In [45]:
# ============================================================
# RECOMMENDATION SIMILARITY STRENGTH @ 10
# ============================================================

TOP_K = 10

similarity_records = []

for query_index in eligible_query_indices:

    query_book_id = nlp_index.iloc[
        query_index
    ]["book_id"]

    recommendations = recommend_by_book_id(
        query_book_id,
        n_recommendations=TOP_K
    )

    for _, row in recommendations.iterrows():

        similarity_records.append(
            {
                "query_book_id":
                    query_book_id,

                "recommended_book_id":
                    row["book_id"],

                "recommendation_rank":
                    row["recommendation_rank"],

                "similarity_score":
                    row["similarity_score"]
            }
        )


similarity_evaluation_df = pd.DataFrame(
    similarity_records
)


# ------------------------------------------------------------
# Overall similarity statistics
# ------------------------------------------------------------

similarity_summary = (
    similarity_evaluation_df[
        "similarity_score"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95
        ]
    )
)


print("RECOMMENDATION SIMILARITY STRENGTH @ 10")
print("=" * 80)

print(
    "Recommendation pairs evaluated:",
    len(similarity_evaluation_df)
)

print("\nSimilarity score distribution:")
display(
    similarity_summary.to_frame(
        name="similarity_score"
    )
)


# ------------------------------------------------------------
# Average similarity by recommendation rank
# ------------------------------------------------------------

similarity_by_rank = (
    similarity_evaluation_df
    .groupby(
        "recommendation_rank"
    )["similarity_score"]
    .agg(
        ["count", "mean", "median"]
    )
    .reset_index()
)


print("\nAVERAGE SIMILARITY BY RECOMMENDATION RANK")
print("=" * 80)

display(similarity_by_rank)

RECOMMENDATION SIMILARITY STRENGTH @ 10
Recommendation pairs evaluated: 19396

Similarity score distribution:


,similarity_score
count,19396.000000
mean,0.321981
std,0.144285
min,0.028597
25%,0.224503
50%,0.301556
75%,0.396599
90%,0.507504
95%,0.586490
max,1.000000



AVERAGE SIMILARITY BY RECOMMENDATION RANK


,recommendation_rank,count,mean,median
0,1,2040,0.513143,0.494901
1,2,2026,0.410047,0.396190
2,3,1994,0.361421,0.348542
3,4,1976,0.327658,0.321637
4,5,1961,0.300902,0.292859
5,6,1933,0.282737,0.275497
6,7,1907,0.266633,0.259024
7,8,1876,0.254240,0.247294
8,9,1851,0.243307,0.236033
9,10,1832,0.233117,0.225112


In [46]:
# ============================================================
# TOPIC DIVERSITY OF RECOMMENDATIONS @ 10
# ============================================================

TOP_K = 10

diversity_records = []

for query_index in eligible_query_indices:

    query_book_id = nlp_index.iloc[
        query_index
    ]["book_id"]

    recommendations = recommend_by_book_id(
        query_book_id,
        n_recommendations=TOP_K
    )

    recommendations = enrich_recommendations(
        recommendations
    )

    # Only recommendations with cluster metadata
    valid_clusters = (
        recommendations["topic_cluster"]
        .dropna()
    )

    n_recommendations = len(
        recommendations
    )

    n_clustered = len(
        valid_clusters
    )

    unique_clusters = (
        valid_clusters.nunique()
    )

    cluster_diversity_ratio = (
        unique_clusters / n_clustered
        if n_clustered > 0
        else np.nan
    )

    diversity_records.append(
        {
            "query_book_id":
                query_book_id,

            "recommendations_returned":
                n_recommendations,

            "recommendations_with_cluster":
                n_clustered,

            "unique_topic_clusters":
                unique_clusters,

            "cluster_diversity_ratio":
                cluster_diversity_ratio
        }
    )


recommendation_diversity_df = pd.DataFrame(
    diversity_records
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("RECOMMENDATION TOPIC DIVERSITY @ 10")
print("=" * 80)

print(
    "Queries evaluated:",
    len(recommendation_diversity_df)
)

print(
    "Mean unique topic clusters:",
    round(
        recommendation_diversity_df[
            "unique_topic_clusters"
        ].mean(),
        3
    )
)

print(
    "Median unique topic clusters:",
    recommendation_diversity_df[
        "unique_topic_clusters"
    ].median()
)

print(
    "Mean cluster diversity ratio:",
    round(
        recommendation_diversity_df[
            "cluster_diversity_ratio"
        ].mean(),
        3
    )
)

print(
    "Median cluster diversity ratio:",
    round(
        recommendation_diversity_df[
            "cluster_diversity_ratio"
        ].median(),
        3
    )
)


print("\nDIVERSITY DISTRIBUTION")
print("=" * 80)

display(
    recommendation_diversity_df[
        [
            "unique_topic_clusters",
            "cluster_diversity_ratio"
        ]
    ].describe()
)

RECOMMENDATION TOPIC DIVERSITY @ 10
Queries evaluated: 2040
Mean unique topic clusters: 3.841
Median unique topic clusters: 4.0
Mean cluster diversity ratio: 0.467
Median cluster diversity ratio: 0.444

DIVERSITY DISTRIBUTION


,unique_topic_clusters,cluster_diversity_ratio
count,2040.000000,2040.000000
mean,3.840686,0.467305
std,1.989085,0.259546
min,1.000000,0.100000
25%,2.000000,0.222222
50%,4.000000,0.444444
75%,5.000000,0.666667
max,10.000000,1.000000


In [47]:
# ============================================================
# RECOMMENDATION SOURCE EXPOSURE @ 10
# ============================================================

TOP_K = 10

source_records = []

for query_index in eligible_query_indices:

    query_book_id = nlp_index.iloc[
        query_index
    ]["book_id"]

    query_source = nlp_index.iloc[
        query_index
    ]["source_group"]

    recommendations = recommend_by_book_id(
        query_book_id,
        n_recommendations=TOP_K
    )

    for _, row in recommendations.iterrows():

        source_records.append(
            {
                "query_book_id":
                    query_book_id,

                "query_source":
                    query_source,

                "recommended_book_id":
                    row["book_id"],

                "recommended_source":
                    row["source_group"],

                "similarity_score":
                    row["similarity_score"]
            }
        )


source_exposure_df = pd.DataFrame(
    source_records
)


# ------------------------------------------------------------
# Catalog source distribution
# ------------------------------------------------------------

catalog_source_distribution = (
    nlp_index["source_group"]
    .value_counts()
    .rename_axis("source_group")
    .reset_index(name="catalog_books")
)

catalog_source_distribution[
    "catalog_pct"
] = (
    catalog_source_distribution[
        "catalog_books"
    ]
    / len(nlp_index)
    * 100
)


# ------------------------------------------------------------
# Recommendation source distribution
# ------------------------------------------------------------

recommendation_source_distribution = (
    source_exposure_df[
        "recommended_source"
    ]
    .value_counts()
    .rename_axis("source_group")
    .reset_index(
        name="recommendation_count"
    )
)

recommendation_source_distribution[
    "recommendation_pct"
] = (
    recommendation_source_distribution[
        "recommendation_count"
    ]
    / len(source_exposure_df)
    * 100
)


# ------------------------------------------------------------
# Compare catalog vs recommendation exposure
# ------------------------------------------------------------

source_comparison = (
    catalog_source_distribution
    .merge(
        recommendation_source_distribution,
        on="source_group",
        how="outer"
    )
    .fillna(0)
)

source_comparison[
    "exposure_difference_pct_points"
] = (
    source_comparison[
        "recommendation_pct"
    ]
    -
    source_comparison[
        "catalog_pct"
    ]
)


print("SOURCE EXPOSURE IN RECOMMENDATIONS @ 10")
print("=" * 80)

display(
    source_comparison.sort_values(
        "catalog_books",
        ascending=False
    )
)


# ------------------------------------------------------------
# Query-source → recommendation-source relationship
# ------------------------------------------------------------

source_transition = pd.crosstab(
    source_exposure_df[
        "query_source"
    ],
    source_exposure_df[
        "recommended_source"
    ],
    normalize="index"
) * 100


print(
    "\nQUERY SOURCE → RECOMMENDED SOURCE (%)"
)
print("=" * 80)

display(
    source_transition.round(2)
)

SOURCE EXPOSURE IN RECOMMENDATIONS @ 10


,source_group,catalog_books,catalog_pct,recommendation_count,recommendation_pct,exposure_difference_pct_points
1,LeadershipNow only,1117,54.039671,8774,45.236131,-8.803540
2,Open Library only,947,45.815191,10600,54.650443,8.835252
0,Both,3,0.145138,22,0.113425,-0.031712



QUERY SOURCE → RECOMMENDED SOURCE (%)


recommended_source,Both,LeadershipNow only,Open Library only
query_source,,,
Both,0.00,13.33,86.67
LeadershipNow only,0.02,79.25,20.73
Open Library only,0.21,9.61,90.17


In [49]:
# ============================================================
# CHECK AVAILABLE PROJECT / MODEL PATH VARIABLES
# ============================================================

path_variables = {
    name: value
    for name, value in globals().items()
    if (
        "path" in name.lower()
        or "root" in name.lower()
        or "model" in name.lower()
    )
}

for name, value in path_variables.items():
    print(f"{name}: {value}")

Path: <class 'pathlib._local.Path'>
MODELS_DIR: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/models
CLUSTER_BOOKS_PATH: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/books_with_final_topic_clusters.csv
NLP_INDEX_PATH: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/nlp_book_index.csv
ENRICHED_MATRIX_PATH: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/models/enriched_tfidf_matrix.npz
ENRICHED_VECTORIZER_PATH: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/models/enriched_tfidf_vectorizer.joblib


In [50]:
# ============================================================
# LOAD CORE TF-IDF MATRIX
# FOR SOURCE-SENSITIVITY COMPARISON
# ============================================================

from scipy.sparse import load_npz

CORE_MATRIX_PATH = (
    MODELS_DIR
    / "core_tfidf_matrix.npz"
)

core_tfidf = load_npz(
    CORE_MATRIX_PATH
)


# ------------------------------------------------------------
# Validate alignment
# ------------------------------------------------------------

print("CORE TF-IDF MATRIX VALIDATION")
print("=" * 80)

print(
    "Path:",
    CORE_MATRIX_PATH
)

print(
    "Shape:",
    core_tfidf.shape
)

print(
    "NLP index rows:",
    len(nlp_index)
)

print(
    "Matrix aligned:",
    core_tfidf.shape[0] == len(nlp_index)
)

print(
    "Core zero vectors:",
    (core_tfidf.getnnz(axis=1) == 0).sum()
)

CORE TF-IDF MATRIX VALIDATION
Path: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/models/core_tfidf_matrix.npz
Shape: (2067, 1816)
NLP index rows: 2067
Matrix aligned: True
Core zero vectors: 42


In [51]:
# ============================================================
# CORE VS ENRICHED TF-IDF
# SOURCE-SENSITIVITY COMPARISON @ 10
# ============================================================

def recommend_with_matrix(
    book_id,
    tfidf_matrix,
    n_recommendations=10
):
    """
    Generate recommendations using a supplied TF-IDF matrix.

    Uses the same recommendation rules:
    - exclude query book
    - exclude zero similarity
    - suppress same title + author combinations
    """

    matches = nlp_index.index[
        nlp_index["book_id"] == book_id
    ].tolist()

    if not matches:
        return pd.DataFrame()

    query_index = matches[0]

    # Zero-vector query
    if tfidf_matrix[query_index].nnz == 0:
        return pd.DataFrame()

    similarity_scores = cosine_similarity(
        tfidf_matrix[query_index],
        tfidf_matrix
    ).flatten()

    ranked_indices = np.argsort(
        similarity_scores
    )[::-1]

    # Query duplicate key
    query_book = books[
        books["book_id"] == book_id
    ]

    if not query_book.empty:
        query_key = (
            query_book.iloc[0]["recommendation_title_key"],
            query_book.iloc[0]["recommendation_author_key"]
        )
    else:
        query_key = None

    selected_indices = []
    seen_keys = set()

    for idx in ranked_indices:

        if idx == query_index:
            continue

        if similarity_scores[idx] <= 0:
            continue

        candidate_id = nlp_index.iloc[idx]["book_id"]

        candidate_book = books[
            books["book_id"] == candidate_id
        ]

        if not candidate_book.empty:

            candidate_key = (
                candidate_book.iloc[0]["recommendation_title_key"],
                candidate_book.iloc[0]["recommendation_author_key"]
            )

            if (
                query_key is not None
                and candidate_key == query_key
            ):
                continue

            if candidate_key in seen_keys:
                continue

            seen_keys.add(candidate_key)

        selected_indices.append(idx)

        if len(selected_indices) >= n_recommendations:
            break

    recommendations = (
        nlp_index
        .iloc[selected_indices]
        .copy()
    )

    recommendations["similarity_score"] = (
        similarity_scores[selected_indices]
    )

    return recommendations


# ============================================================
# SOURCE EXPOSURE EVALUATION FUNCTION
# ============================================================

def evaluate_source_behavior(
    tfidf_matrix,
    model_name,
    top_k=10
):
    records = []

    eligible_indices = np.where(
        tfidf_matrix.getnnz(axis=1) > 0
    )[0]

    for query_index in eligible_indices:

        query_book_id = nlp_index.iloc[
            query_index
        ]["book_id"]

        query_source = nlp_index.iloc[
            query_index
        ]["source_group"]

        recommendations = recommend_with_matrix(
            query_book_id,
            tfidf_matrix,
            n_recommendations=top_k
        )

        for _, row in recommendations.iterrows():

            records.append(
                {
                    "model": model_name,
                    "query_source": query_source,
                    "recommended_source":
                        row["source_group"]
                }
            )

    result_df = pd.DataFrame(records)

    transition = pd.crosstab(
        result_df["query_source"],
        result_df["recommended_source"],
        normalize="index"
    ) * 100

    return result_df, transition


# ============================================================
# RUN BOTH MODELS
# ============================================================

core_source_df, core_transition = (
    evaluate_source_behavior(
        core_tfidf,
        "Core",
        top_k=10
    )
)

enriched_source_df, enriched_transition = (
    evaluate_source_behavior(
        enriched_tfidf,
        "Enriched",
        top_k=10
    )
)


print("CORE TF-IDF")
print("=" * 80)
display(core_transition.round(2))

print("\nENRICHED TF-IDF")
print("=" * 80)
display(enriched_transition.round(2))

CORE TF-IDF


recommended_source,Both,LeadershipNow only,Open Library only
query_source,,,
Both,0.00,33.33,66.67
LeadershipNow only,0.15,75.72,24.13
Open Library only,0.11,17.47,82.43



ENRICHED TF-IDF


recommended_source,Both,LeadershipNow only,Open Library only
query_source,,,
Both,0.00,13.33,86.67
LeadershipNow only,0.02,79.25,20.73
Open Library only,0.21,9.61,90.17


In [52]:
# ============================================================
# CORE VS ENRICHED RECOMMENDATION COMPARISON
# ============================================================

comparison_records = []

# Only compare books that have usable vectors
# in BOTH representations
common_query_indices = np.where(
    (core_tfidf.getnnz(axis=1) > 0)
    &
    (enriched_tfidf.getnnz(axis=1) > 0)
)[0]


for query_index in common_query_indices:

    query_book_id = nlp_index.iloc[
        query_index
    ]["book_id"]

    # --------------------------------------------------------
    # Core recommendations
    # --------------------------------------------------------

    core_rec = recommend_with_matrix(
        query_book_id,
        core_tfidf,
        n_recommendations=10
    )

    # --------------------------------------------------------
    # Enriched recommendations
    # --------------------------------------------------------

    enriched_rec = recommend_with_matrix(
        query_book_id,
        enriched_tfidf,
        n_recommendations=10
    )

    core_ids = set(
        core_rec["book_id"].tolist()
    )

    enriched_ids = set(
        enriched_rec["book_id"].tolist()
    )

    # --------------------------------------------------------
    # Top-10 overlap
    # --------------------------------------------------------

    overlap_count = len(
        core_ids.intersection(
            enriched_ids
        )
    )

    overlap_ratio = (
        overlap_count / 10
    )

    comparison_records.append(
        {
            "book_id":
                query_book_id,

            "core_recommendations":
                len(core_rec),

            "enriched_recommendations":
                len(enriched_rec),

            "core_mean_similarity":
                core_rec["similarity_score"].mean()
                if len(core_rec) > 0
                else np.nan,

            "enriched_mean_similarity":
                enriched_rec["similarity_score"].mean()
                if len(enriched_rec) > 0
                else np.nan,

            "core_rank1_similarity":
                core_rec["similarity_score"].iloc[0]
                if len(core_rec) > 0
                else np.nan,

            "enriched_rank1_similarity":
                enriched_rec["similarity_score"].iloc[0]
                if len(enriched_rec) > 0
                else np.nan,

            "top10_overlap_count":
                overlap_count,

            "top10_overlap_ratio":
                overlap_ratio
        }
    )


core_enriched_comparison = pd.DataFrame(
    comparison_records
)


# ============================================================
# SUMMARY
# ============================================================

print("CORE VS ENRICHED RECOMMENDATION COMPARISON")
print("=" * 80)

print(
    "Queries compared:",
    len(core_enriched_comparison)
)

summary = pd.DataFrame(
    {
        "Metric": [
            "Mean Top-10 similarity",
            "Mean Rank-1 similarity",
            "Mean recommendations returned"
        ],

        "Core": [
            core_enriched_comparison[
                "core_mean_similarity"
            ].mean(),

            core_enriched_comparison[
                "core_rank1_similarity"
            ].mean(),

            core_enriched_comparison[
                "core_recommendations"
            ].mean()
        ],

        "Enriched": [
            core_enriched_comparison[
                "enriched_mean_similarity"
            ].mean(),

            core_enriched_comparison[
                "enriched_rank1_similarity"
            ].mean(),

            core_enriched_comparison[
                "enriched_recommendations"
            ].mean()
        ]
    }
)

display(summary)


print("\nTOP-10 MODEL AGREEMENT")
print("=" * 80)

print(
    "Mean shared recommendations:",
    round(
        core_enriched_comparison[
            "top10_overlap_count"
        ].mean(),
        3
    )
)

print(
    "Median shared recommendations:",
    core_enriched_comparison[
        "top10_overlap_count"
    ].median()
)

print(
    "Mean Top-10 overlap ratio:",
    round(
        core_enriched_comparison[
            "top10_overlap_ratio"
        ].mean(),
        3
    )
)

CORE VS ENRICHED RECOMMENDATION COMPARISON
Queries compared: 2025


,Metric,Core,Enriched
0,Mean Top-10 similarity,0.415013,0.324828
1,Mean Rank-1 similarity,0.629950,0.515557
2,Mean recommendations returned,9.240000,9.553086



TOP-10 MODEL AGREEMENT
Mean shared recommendations: 5.59
Median shared recommendations: 6.0
Mean Top-10 overlap ratio: 0.559


In [53]:
# ============================================================
# CORE VS ENRICHED CATALOG COVERAGE @ 10
# ============================================================

def evaluate_catalog_coverage(
    tfidf_matrix,
    top_k=10
):
    eligible_indices = np.where(
        tfidf_matrix.getnnz(axis=1) > 0
    )[0]

    recommended_ids = set()
    total_slots = 0

    for query_index in eligible_indices:

        query_book_id = nlp_index.iloc[
            query_index
        ]["book_id"]

        recommendations = recommend_with_matrix(
            query_book_id,
            tfidf_matrix,
            n_recommendations=top_k
        )

        recommended_ids.update(
            recommendations[
                "book_id"
            ].tolist()
        )

        total_slots += len(
            recommendations
        )

    return {
        "eligible_queries":
            len(eligible_indices),

        "total_recommendations":
            total_slots,

        "unique_books_recommended":
            len(recommended_ids),

        "catalog_coverage_pct":
            len(recommended_ids)
            / len(nlp_index)
            * 100
    }


core_coverage = evaluate_catalog_coverage(
    core_tfidf,
    top_k=10
)

enriched_coverage = evaluate_catalog_coverage(
    enriched_tfidf,
    top_k=10
)


coverage_comparison = pd.DataFrame(
    [
        {
            "representation": "Core TF-IDF",
            **core_coverage
        },
        {
            "representation": "Enriched TF-IDF",
            **enriched_coverage
        }
    ]
)


print("CORE VS ENRICHED CATALOG COVERAGE @ 10")
print("=" * 80)

display(
    coverage_comparison
)

CORE VS ENRICHED CATALOG COVERAGE @ 10


,representation,eligible_queries,total_recommendations,unique_books_recommended,catalog_coverage_pct
0,Core TF-IDF,2025,18711,1995,96.516691
1,Enriched TF-IDF,2040,19396,2016,97.532656


In [54]:
# ============================================================
# SAVE RECOMMENDATION SYSTEM EVALUATION OUTPUTS
# ============================================================

RECOMMENDER_OUTPUT_DIR = (
    MODELS_DIR.parent
    / "data"
    / "processed"
)

RECOMMENDER_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 1. Similarity evaluation
# ------------------------------------------------------------

similarity_evaluation_df.to_csv(
    RECOMMENDER_OUTPUT_DIR
    / "recommendation_similarity_evaluation.csv",
    index=False
)


# ------------------------------------------------------------
# 2. Similarity by rank
# ------------------------------------------------------------

similarity_by_rank.to_csv(
    RECOMMENDER_OUTPUT_DIR
    / "recommendation_similarity_by_rank.csv",
    index=False
)


# ------------------------------------------------------------
# 3. Topic diversity
# ------------------------------------------------------------

recommendation_diversity_df.to_csv(
    RECOMMENDER_OUTPUT_DIR
    / "recommendation_topic_diversity.csv",
    index=False
)


# ------------------------------------------------------------
# 4. Source exposure
# ------------------------------------------------------------

source_comparison.to_csv(
    RECOMMENDER_OUTPUT_DIR
    / "recommendation_source_exposure.csv",
    index=False
)


# ------------------------------------------------------------
# 5. Core vs Enriched comparison
# ------------------------------------------------------------

core_enriched_comparison.to_csv(
    RECOMMENDER_OUTPUT_DIR
    / "core_vs_enriched_recommendations.csv",
    index=False
)


# ------------------------------------------------------------
# 6. Coverage comparison
# ------------------------------------------------------------

coverage_comparison.to_csv(
    RECOMMENDER_OUTPUT_DIR
    / "core_vs_enriched_coverage.csv",
    index=False
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

saved_files = [
    "recommendation_similarity_evaluation.csv",
    "recommendation_similarity_by_rank.csv",
    "recommendation_topic_diversity.csv",
    "recommendation_source_exposure.csv",
    "core_vs_enriched_recommendations.csv",
    "core_vs_enriched_coverage.csv"
]

print("RECOMMENDATION EVALUATION ARTIFACTS")
print("=" * 80)

for filename in saved_files:
    path = RECOMMENDER_OUTPUT_DIR / filename

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{filename}"
    )

RECOMMENDATION EVALUATION ARTIFACTS
✓ recommendation_similarity_evaluation.csv
✓ recommendation_similarity_by_rank.csv
✓ recommendation_topic_diversity.csv
✓ recommendation_source_exposure.csv
✓ core_vs_enriched_recommendations.csv
✓ core_vs_enriched_coverage.csv


# Recommendation System — Final Summary

## Objective

This notebook developed and evaluated a content-based recommendation system for the Leadership and Management Book Recommendation System.

The final recommender uses the enriched TF-IDF representation created during NLP processing:

**Title + Authors + Subjects + Description**

Book-to-book similarity is calculated using **cosine similarity** directly on the original sparse TF-IDF matrix.

The previously developed topic clusters are retained as descriptive metadata and are not used to force recommendation membership.

---

## Recommendation Architecture

The final recommendation workflow is:

1. User searches for a book title.
2. The system identifies the corresponding canonical book record.
3. The book's enriched TF-IDF vector is retrieved.
4. Cosine similarity is calculated against the book catalog.
5. The query book itself is excluded.
6. Zero-similarity candidates are excluded.
7. Repeated normalized title + author combinations are suppressed.
8. Candidates are ranked by descending cosine similarity.
9. The Top-N recommendations are returned with book and topic metadata.

This approach calculates similarity on demand rather than storing a complete book-to-book similarity matrix.

---

## Technical Validation

Representative recommendation tests were performed across:

- General Leadership
- Emotional Intelligence
- Project Management
- Servant Leadership
- Change Management
- Human Resource Management
- Strategy

All technical integrity checks passed:

- Query book excluded
- Unique book IDs returned
- Positive similarity scores only
- Similarities correctly ranked in descending order
- Requested Top-N limit respected

Qualitative inspection also showed meaningful semantic relationships across the tested management and leadership domains.

---

## Catalog Coverage

Using the enriched TF-IDF representation:

- Total catalog: **2,067 books**
- Eligible non-zero-vector queries: **2,040**
- Recommendation pairs generated at Top-10: **19,396**
- Unique books appearing in recommendations: **2,016**
- Catalog Coverage@10: **97.53%**

The high catalog coverage indicates that recommendation exposure is distributed across most of the available catalog rather than being restricted to a small subset of books.

---

## Recommendation Similarity

Across 19,396 recommendation pairs:

- Mean cosine similarity: **0.322**
- Median cosine similarity: **0.302**
- 25th percentile: **0.225**
- 75th percentile: **0.397**
- 90th percentile: **0.508**
- 95th percentile: **0.586**

Average similarity declined consistently with recommendation rank:

- Rank 1 mean similarity: **0.513**
- Rank 2: **0.410**
- Rank 3: **0.361**
- Rank 5: **0.301**
- Rank 10: **0.233**

This monotonic decline is consistent with the intended cosine-similarity ranking mechanism.

Absolute cosine scores are interpreted relative to this corpus rather than as probabilities or universal measures of recommendation quality.

---

## Topic Diversity

Across the 2,040 eligible queries:

- Mean unique topic clusters per recommendation list: **3.84**
- Median unique topic clusters: **4**
- Mean cluster diversity ratio: **0.467**
- Median cluster diversity ratio: **0.444**

The recommender therefore maintains topical focus while still allowing recommendations to cross cluster boundaries when supported by textual similarity.

---

## Source Sensitivity

The catalog combines books from Open Library and LeadershipNow.

Recommendation exposure showed source-associated behavior.

### Enriched TF-IDF

- LeadershipNow query → LeadershipNow recommendation: **79.25%**
- Open Library query → Open Library recommendation: **90.17%**

### Core TF-IDF

Using only title + authors:

- LeadershipNow query → LeadershipNow recommendation: **75.72%**
- Open Library query → Open Library recommendation: **82.43%**

The comparison indicates that source-associated structure already exists in the common title-and-author representation, while additional subjects and descriptions strengthen same-source recommendation patterns, particularly for Open Library.

This is treated as a limitation associated with heterogeneous metadata availability rather than evidence of recommendation accuracy or inaccuracy.

---

## Core vs Enriched Representation

A controlled comparison was performed across **2,025 books** with usable vectors in both representations.

Average Top-10 agreement between Core and Enriched TF-IDF was:

- Mean shared recommendations: **5.59 of 10**
- Median shared recommendations: **6 of 10**
- Mean Top-10 overlap ratio: **0.559**

Therefore, adding subjects and descriptions materially changes the recommendation neighborhood rather than merely rescaling similarity.

Catalog coverage was:

- Core TF-IDF: **96.52%**
- Enriched TF-IDF: **97.53%**

The enriched representation also supported more eligible queries and surfaced more unique books.

---

## Final Model Selection

**Enriched TF-IDF + Cosine Similarity** is retained as the primary recommendation engine.

The decision is based on:

- richer semantic representation;
- coherent qualitative recommendations across multiple management domains;
- high catalog coverage;
- broad recommendation availability;
- meaningful topic diversity;
- successful technical integrity validation.

The stronger source-associated recommendation behavior introduced by heterogeneous metadata enrichment is retained as an explicit model limitation.

Topic clusters remain supplementary explanatory metadata rather than hard recommendation constraints.

---

## Limitations

The system does not contain explicit user ratings, clicks, purchases, or relevance judgments. Therefore, conventional supervised recommender metrics such as accuracy, precision, recall, F1-score, RMSE, and MAE are not interpreted as primary performance measures.

Evaluation instead focuses on:

- technical integrity;
- qualitative semantic relevance;
- cosine-similarity behavior;
- catalog coverage;
- topic diversity;
- source sensitivity;
- representation comparison.

Some edition-labelled variants may remain as separate recommendations when they exist as distinct canonical records.

---

## Final Status

**Notebook 12 — Recommendation System: COMPLETE**

Final recommendation engine:

**Enriched TF-IDF → Cosine Similarity → Duplicate Suppression → Ranked Top-N Recommendations**

The recommendation system is ready for integration into the subsequent application and deployment stages.